In [ ]:
import os, sys, random, pandas as pd, numpy as np
from tqdm import tqdm
import torch


# set random seed
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
SEED = 10
random.seed(SEED)
np.random.seed(SEED)
torch.random.manual_seed(SEED)
sys.path.append('../')


# load package requirments
from PACE.utils import  generate_steps, generate_multi_steps, parser, _valid_criterions
from PACE.plots import plot_comparision
from PACE.train import pretrain, train, test
from PACE.models import  PACE, ODEFunc2
from PACE.constants import DATA_DIR
from PACE.exp import setup_exp
from PACE.eval import generate_plot_data
from torchdiffeq import odeint_adjoint as odeint

### Defining parameters and loading dataset

In [ ]:
import torch.optim as optim

dim=203

import sys

# Simulate the command-line arguments
sys.argv = [
    'PACE Training',
    '-d', 'file',
    '-c', 'ot1',
    '-n', 'ipsc_203',
    '--no-hold-one-out',
    '-ho', '-1',
    '-e1', '10000',
    '-e2', '30',
    '-bs', '1024',
    '-r', '0.01',
    '-rm', '0.1',
    '--norm_cost',
    '--n_big_batches', '5',
    '--learning_rate2', '0.0001'
]

args = parser.parse_args()
opts = vars(args)

# Display the parsed arguments
print(opts)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device


In [ ]:
df_all = pd.read_csv(DATA_DIR + '/ipsc_203.csv')
df_all=df_all.iloc[:,:dim+1]
 # make output dir
if not os.path.isdir(opts['output_dir']):
    os.makedirs(opts['output_dir'])
exp_dir, logger = setup_exp(opts['output_dir'], opts, opts['name'])
  # load dataset
logger.info(f'Loading dataset: '+opts['name'])


In [ ]:
df_all.columns=['samples']+['x'+str(i+1) for i in range(dim)]
df = df_all

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(df.iloc[:,1],df.iloc[:,2],c=df.iloc[:,0])

In [ ]:
groups = sorted([g for g in df.samples.unique() if g != args.hold_out])
steps = generate_steps(groups)
pace = PACE(in_out_dim=dim, hidden_dim=512, n_hiddens=5, activation='leakyrelu', num_time_stamps = len(steps))
optimizer = torch.optim.Adam(pace.parameters())
opts['criterion']='ot1'
criterion =  _valid_criterions[opts['criterion']]()

# # logger.info(f'Extracting parameters')
use_cuda = torch.cuda.is_available()
# sample_with_replacement = opts['sample_with_replacement' ]
# apply_losses_in_time = opts['apply_losses_in_time']

    
local_losses = {f'{t0}:{t1}':[] for (t0, t1) in steps}
batch_losses = []
globe_losses = []

In [ ]:
pace=pace.to(device)

In [ ]:
initial_size=df[df['samples']==0].x1.shape[0]
initial_size

In [ ]:
sample_sizes = df.groupby('samples').size()
ref0 = sample_sizes / sample_sizes.iloc[0]  
relative_mass = torch.tensor(ref0.values) 
relative_mass

In [ ]:
sample_size = (np.round(1/args.n_big_batches*(len(df[df['samples']==0]))).astype(int),)
sample_size

### Training with $\mathcal{L}_{\rm VGFM}$

In [ ]:
pace, v_losses, g_losses, losses = pretrain(
    pace,
    df,
    optimizer,
    steps,
    # n_epoch=1,
    n_epoch=args.epochs1,
    hold_out=args.hold_out,
    logger=logger,
    relative_mass=relative_mass,
    reg=args.reg,
    reg_m=[args.reg_m, np.inf],
    norm_cost=args.norm_cost,
    batch_size=args.n_batch
)


In [ ]:
import matplotlib.pyplot as plt
plt.plot(losses,label='total')
plt.plot(v_losses,label='velocity')
plt.plot(g_losses,label='growth')
plt.legend()

In [ ]:
torch.save(pace.state_dict(), os.path.join(exp_dir+'/pretrain_best_model'))

In [ ]:
pace.load_state_dict(torch.load(os.path.join(exp_dir+'/pretrain_best_model'),map_location=torch.device('cpu')))
optimizer = torch.optim.Adam(pace.parameters(),lr=args.learning_rate1)

### Finetuning $\mathcal{L}_{\rm multiscale}$

In [ ]:
multi_steps = generate_multi_steps(groups)
multi_steps

In [ ]:
local_losses = {f'{t0}:{t1}':[] for (t0, t1) in multi_steps}

In [ ]:
pace=pace.to(device)

In [ ]:
pace, v_losses, g_losses, losses = pretrain(
    pace,
    df,
    optimizer,
    multi_steps,
    n_epoch=args.epochs1,
    # n_epoch=1,
    hold_out=args.hold_out,
    logger=logger,
    relative_mass=relative_mass,
    reg=args.reg,
    reg_m=[args.reg_m, np.inf],
    norm_cost=args.norm_cost,
    batch_size=args.n_batch,
    stage='finetune'
)


In [ ]:
torch.save(pace.state_dict(), os.path.join(exp_dir+'/finetune_best_model'))

In [ ]:
pace.load_state_dict(torch.load(os.path.join(exp_dir+'/finetune_best_model'),map_location=torch.device('cpu')),strict=False)
optimizer = torch.optim.Adam(pace.parameters(),lr=args.learning_rate2)

### Incorporating $\mathcal{L}_{\rm OT}$

In [ ]:
logger.info("Incorporating $L_{ot}$")

# Training phase
l_loss, b_loss, g_loss = train(
    pace,
    df,
    groups,
    multi_steps,
    optimizer,
    n_epoch = args.epochs2,  # n_epoch
    criterion=criterion,
    use_cuda=use_cuda,
    apply_losses_in_time=args.apply_losses_in_time,
    hold_one_out=args.hold_one_out,
    hold_out=args.hold_out,
    sample_size=sample_size,
    relative_mass=relative_mass,
    initial_size=initial_size,
    sample_with_replacement=args.sample_with_replacement,
    logger=logger,
    device=device,
    best_model_path=exp_dir + "/best_model",
    stepsize=0.1,
    n_big_batch=args.n_big_batches
)

# Accumulate losses
for k, v in l_loss.items():
    local_losses[k].extend(v)

batch_losses.extend(b_loss)
globe_losses.extend(g_loss)


In [ ]:
import os
import matplotlib.pyplot as plt

# Assuming you have variables: l_loss and exp_dir
plt.plot(l_loss['0:1'], label='t1')
plt.legend()

# Concatenate save path
save_path = os.path.join(exp_dir, 'emd_train_each_tp.png')
plt.savefig(save_path)

### Test

In [ ]:
pace.load_state_dict(torch.load(os.path.join(exp_dir+'/best_model'),map_location=torch.device('cpu')))

In [ ]:
seeds = [0, 1, 2, 3, 4]

from collections import defaultdict
final_metrics = defaultdict(lambda: {
    "RME": [],
    "OT": [],
    "PCC": [],
    "DeltaCos":[],
    "DeltaSign":[],
    "DeltaPCC":[],
})

for seed in seeds:
    result = test(
        pace, df, groups, multi_steps, optimizer, 1,
        criterion=criterion,
        use_cuda=use_cuda,
        apply_losses_in_time=False,
        hold_one_out=args.hold_one_out,
        hold_out=args.hold_out,
        sample_size=sample_size,
        relative_mass=relative_mass,
        sample_with_replacement=args.sample_with_replacement,
        logger=logger,
        device=device,
        stepsize=0.1,
        test=True,
        seed=seed
    )

    for k, v in result.items():

        final_metrics[k]["RME"].append(v["RME"][0])
        final_metrics[k]["OT"].append(v["OT"][0])
        final_metrics[k]["PCC"].append(v["PCC"][0])
        final_metrics[k]["DeltaCos"].append(v["DeltaCos"][0])
        final_metrics[k]["DeltaSign"].append(v["DeltaSign"][0])
        final_metrics[k]["DeltaPCC"].append(v["DeltaPCC"][0])

for k, v in final_metrics.items():
    print(
        f"{k} | "
        f"RME: {np.mean(v['RME']):.4f} ± {np.std(v['RME']):.4f} | "
        f"OT:  {np.mean(v['OT']):.4f} ± {np.std(v['OT']):.4f} | "
        f"PCC: {np.mean(v['PCC']):.4f} ± {np.std(v['PCC']):.4f}|"
        f"DeltaCos: {np.mean(v['DeltaCos']):.4f} ± {np.std(v['DeltaCos']):.4f} | "
        f"DeltaSign:  {np.mean(v['DeltaSign']):.4f} ± {np.std(v['DeltaSign']):.4f} | "
        f"DeltaPCC: {np.mean(v['DeltaPCC']):.4f} ± {np.std(v['DeltaPCC']):.4f}"
    )

## Deeper analysis

In [ ]:
pace.load_state_dict(torch.load(os.path.join(exp_dir+'/best_model'),map_location=torch.device('cpu')))

### Analysis on mean and variance

In [ ]:
# generate plot data
pace.to('cpu')
# generated, trajectories,_,_ = generate_plot_data(
generated = generate_plot_data(
    pace, df, multi_steps, n_points=initial_size, n_trajectories=50, n_bins=100, 
    sample_with_replacement=False, use_cuda=False, samples_key='samples',
    logger=logger
    )


In [ ]:
np.save(exp_dir+'/generated_final.npy', generated)

In [ ]:
generated = np.load(exp_dir+'/generated_final.npy')
generated_finetune = np.load(exp_dir+'/generated_finetune.npy')
generate_pretrain = np.load(exp_dir+'/generated_pretrain.npy')

In [ ]:
from scipy.stats import linregress
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_mean_var_scatter(generated: np.ndarray, df: pd.DataFrame, timepoint: int = 1, save_dir=None,name=None):
    """
    Scatter plot of log-transformed mean vs variance for real and generated data
    at a specific timepoint. Includes regression lines for each.
    
    Args:
        generated (np.ndarray): Generated data array, shape (time, samples, features)
        df (pd.DataFrame): Real data DataFrame with 'samples' column
        timepoint (int): Time point to plot
    """
    # Extract generated data for the specific timepoint
    gen_vals = generated[timepoint, :, :]
    gen_mean = gen_vals.mean(axis=0)
    gen_var = gen_vals.var(axis=0)

    # Extract real data for the specific timepoint
    real_vals = df[df['samples'] == timepoint].iloc[:, 1:].values
    real_mean = real_vals.mean(axis=0)
    real_var = real_vals.var(axis=0)

    # Log1p transform
    log_gen_mean = np.log1p(np.maximum(gen_mean, -0.9999))
    log_gen_var = np.log1p(np.maximum(gen_var, -0.9999))
    log_real_mean = np.log1p(real_mean)
    log_real_var = np.log1p(real_var)
    # if np.any(np.isnan(log_gen_mean)) or np.any(np.isnan(log_gen_var)):
    #     print("Warning: NaN exists in the generated data!")
    # if np.any(np.isinf(log_gen_mean)) or np.any(np.isinf(log_gen_var)):
    #     print("Warning: Inf exists in the generated data!")
    # Scatter plot
    plt.figure(figsize=(6, 5))
    plt.scatter(log_real_mean, log_real_var, alpha=0.5, label='Real', color='blue')
    plt.scatter(log_gen_mean, log_gen_var, alpha=0.5, label='Generated', color='orange')

    # Add regression lines
    slope_r, intercept_r, *_ = linregress(log_real_mean, log_real_var)
    slope_g, intercept_g, *_ = linregress(log_gen_mean, log_gen_var)
    x_vals = np.linspace(min(log_real_mean.min(), log_gen_mean.min()),
                         max(log_real_mean.max(), log_gen_mean.max()), 100)
    plt.plot(x_vals, slope_r * x_vals + intercept_r, color='blue', linestyle='--')
    plt.plot(x_vals, slope_g * x_vals + intercept_g, color='orange', linestyle='--')

    # Labels and title
    plt.title(f'Timepoint {timepoint}')
    plt.xlabel('log(Mean + 1)')
    plt.ylabel('log(Variance + 1)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'mean_var_scatter_{name}.png'), dpi=300)

# Example usage
plot_mean_var_scatter(generated, df, timepoint=1,save_dir=exp_dir,name='final')
plot_mean_var_scatter(generated_finetune, df, timepoint=1,save_dir=exp_dir,name='finetune')
plot_mean_var_scatter(generate_pretrain, df, timepoint=1,save_dir=exp_dir,name='pretrain')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

def plot_mean_var_histograms_kde_save(generated, df, save_dir="plots",name=None):
    """
    Plot histograms and KDEs of feature-wise mean and variance for real and generated data
    across multiple time points, and save the figures as PNG files.

    Args:
        generated (np.ndarray): Generated data array, shape (time, samples, features)
        df (pd.DataFrame): Real data DataFrame with a 'samples' column
        save_dir (str): Directory to save the figures
    """
    time_points = df["samples"].unique()
    num_timepoints = len(time_points)

    # Create directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)

    for i, t in enumerate(time_points):
        real_vals = df[df["samples"] == t].filter(like='x').values
        gen_vals = generated[i]

        real_mean = real_vals.mean(axis=0)
        gen_mean = gen_vals.mean(axis=0)
        real_var = real_vals.var(axis=0)
        gen_var = gen_vals.var(axis=0)

        # Create figure
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # --- Mean subplot ---
        ax_mean = axes[0]
        ax_mean.hist(real_mean, bins=50, alpha=0.4, label='Real', color='blue', density=True)
        ax_mean.hist(gen_mean, bins=50, alpha=0.4, label='Generated', color='orange', density=True)
        sns.kdeplot(real_mean, ax=ax_mean, color='blue', linewidth=2)
        sns.kdeplot(gen_mean, ax=ax_mean, color='orange', linewidth=2)
        ax_mean.set_title(f'Time {t} - Mean')
        ax_mean.legend()

        # --- Variance subplot ---
        ax_var = axes[1]
        ax_var.hist(real_var, bins=50, alpha=0.4, label='Real', color='blue', density=True)
        ax_var.hist(gen_var, bins=50, alpha=0.4, label='Generated', color='orange', density=True)
        sns.kdeplot(real_var, ax=ax_var, color='blue', linewidth=2)
        sns.kdeplot(gen_var, ax=ax_var, color='orange', linewidth=2)
        ax_var.set_title(f'Time {t} - Variance')
        ax_var.legend()

        plt.tight_layout()

        # Save figure
        fig_path = os.path.join(save_dir, f"time_{t}_mean_var_{name}.png")
        plt.savefig(fig_path, dpi=300)
        # plt.close(fig)  # Close figure to free memory



# Remove the first time point
timepoints = sorted(df['samples'].unique())
df_wo_first = df[df['samples'].isin(timepoints[1:])]
generated_wo_first = generated[1:, :, :]
generated_pretrain_wo_first = generate_pretrain[1:, :, :]
generated_finetune_wo_first = generated_finetune[1:, :, :]
# Plot histograms and KDEs for the remaining time points
plot_mean_var_histograms_kde_save(generated_wo_first, df_wo_first,save_dir=exp_dir,name='final')
plot_mean_var_histograms_kde_save(generated_finetune_wo_first, df_wo_first,save_dir=exp_dir,name='finetune')
plot_mean_var_histograms_kde_save(generated_pretrain_wo_first, df_wo_first,save_dir=exp_dir,name='pretrain')


### Capturing Rate Heterogeneity

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

def extract_module_data(model, device='cpu'):
    """
    Data extraction function specifically for Case Studies
    """
    model.eval()
    model.to(device)
    
    results = {}
    
    # --- 1. Extract t_weights (static parameters) ---
    # shape: [num_time_stamps, hidden_dim]
    t_weights_data = model.t_weights.detach().cpu().numpy()
    results['t_weights'] = t_weights_data
    
    return results

def extract_fusion_pattern(model, t_pair, device='cpu'):
    """
    Input a specific time pair (t_start, t_end) to observe the activation of fusion_weights
    """
    model.eval()
    model.to(device)
    
    # Construct input tensor [1, 2]
    tt = torch.tensor([t_pair], dtype=torch.float32).to(device)
    
    with torch.no_grad():
        # Simulate the logic in the forward pass
        # 1. Pass through the network
        fusion_out = model.fusion_weights(tt) # [1, num_time * hidden]
        
        # 2. Reshape
        B = fusion_out.shape[0]
        fusion_out = fusion_out.view(B, model.num_time_stamps, model.hidden_dim)
        
        # 3. Softmax (obtain final gating weights)
        # shape: [1, num_time_stamps, hidden_dim]
        fusion_probs = torch.softmax(fusion_out, dim=1)
        
    return fusion_probs.squeeze(0).cpu().numpy() # [num_time_stamps, hidden_dim]

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def plot_fusion_heatmap(model, test_pairs=[(0, 1), (0, 2), (0, 3)], device='cpu'):
    """
    Display the weight matrix using a heatmap (lighter red version)
    """
    # 1. Collect data matrix (simulated data)
    data_matrix = []
    labels = []
    
    for i, pair in enumerate(test_pairs):
        # Replace with your actual data retrieval
        activation_map = extract_fusion_pattern(model, pair, device)
        importance_score = activation_map.mean(axis=1)
        
        data_matrix.append(importance_score)
        labels.append(f"t{pair[0]} $\\to$ t{pair[1]}")
    
    data_matrix = np.array(data_matrix)
    
    # --- 2. Create custom "light red" colormap ---
    # h_neg=240 (blue), h_pos=10 (red)
    # s=75 (saturation, standard is 75, decrease to 50 if too vivid)
    # l=65 (lightness, standard is around 50. Setting to 65~70 makes red noticeably pink/lighter)
    custom_cmap = sns.diverging_palette(
        h_neg=250, h_pos=20, 
        s=80, l=65,       # <--- [Key point] Increase l (lightness) to make the color lighter
        as_cmap=True      # Must be set to True to be used as a cmap for the heatmap
    )

    # --- 3. Plotting ---
    plt.figure(figsize=(6, 4))
    
    ax = sns.heatmap(data_matrix, annot=True, fmt=".2f", 
                     cmap=custom_cmap,  # <--- Use custom colormap
                     center=0.33,       # Maintain white center position
                     yticklabels=labels,
                     xticklabels=[f"t{i} $\\to$ t{i+1}" for i in range(3)],
                     cbar_kws={'label': 'Attention Score'},
                     linewidths=2, linecolor='white')
    
    plt.title("Attention Response", fontsize=12)
    plt.ylabel("Input Query", fontsize=10)
    plt.xlabel("Time Interval Indices", fontsize=10)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
# plot_fusion_heatmap(pace)
# For demonstration, just call it directly (as it uses simulated data internally)
plot_fusion_heatmap(pace)

### Analysis of Reconstruction

In [ ]:
import umap
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
# ===============================
# 1. Data
# ===============================
X = df_all.iloc[:, -203:].values
time = df_all['samples'].values   # Time label

# If time is a string, sort first and map to numerical values
unique_times = np.unique(time)
time_to_num = {t: i for i, t in enumerate(sorted(unique_times))}
time_num = np.array([time_to_num[t] for t in time])

combined_points = X
combined_labels  = np.array([0]*1200+[1]*1200+[2]*1200+[3]*1200)

pca = PCA(n_components=12)  # You can adjust the number of components
pca_result = pca.fit_transform(combined_points)

# ===============================
# 2. UMAP
# ===============================
reducer = umap.UMAP(n_neighbors=150, min_dist=0.8,spread=0.8, n_components=2,random_state=0)

umap_emb = reducer.fit_transform(pca_result)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

T, N, D = generated.shape
gen_flat = generated.reshape(-1, D)
gen_emb = reducer.transform(pca.transform(gen_flat))

# ==========================================
# Global style (aligned with your provided plot)
# ==========================================
dpi = 600
point_size = 8
alpha = 1

colorlist = ['#039F89', '#3B5387', '#4EB9D3', '#D2AA3A']

# ==========================================
# 1. Create canvas
# ==========================================
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(12, 5), dpi=dpi
)

# ==========================================
# 2. Figure 1: Original data (colored by time / sample)
# ==========================================
time_orig = df_all['samples'].values
unique_times = sorted(np.unique(time_orig))
time_to_color = {
    t: colorlist[i % len(colorlist)]
    for i, t in enumerate(unique_times)
}

for t in unique_times:
    mask = time_orig == t
    ax1.scatter(
        umap_emb[mask, 0],
        umap_emb[mask, 1],
        c=time_to_color[t],
        s=point_size,
        alpha=alpha,
        label=f'{t} d'
    )

ax1.set_title("Original")
ax1.set_xlabel("UMAP 1")
ax1.set_ylabel("UMAP 2")
ax1.set_xticks([])
ax1.set_yticks([])
for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)
ax1.legend()

# ==========================================
# 3. Figure 2: Generated data (Temporal Evolution)
# ==========================================
gen_emb = gen_emb.reshape(T, N, 2)

for t in range(T):
    ax2.scatter(
        gen_emb[t, :, 0],
        gen_emb[t, :, 1],
        c=colorlist[t % len(colorlist)],
        s=point_size,
        alpha=alpha,
        label=f'{t} d'
    )

ax2.set_title("Generated")
ax2.set_xlabel("UMAP 1")
ax2.set_ylabel("UMAP 2")
ax2.set_xticks([])
ax2.set_yticks([])
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)

ax2.legend()

plt.tight_layout()
plt.savefig(
    "umap.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.show()
